In [ ]:
import numpy as np

In [ ]:
# ==============================================================================
# CELL 0: IMPORTS & CONFIGURATION
# ==============================================================================
import os
import time
import concurrent.futures
import numpy as np
import pandas as pd
import scipy.constants as const
from scipy.special import wofz
from scipy.optimize import curve_fit
from scipy.stats import binned_statistic, gaussian_kde
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from tqdm.auto import tqdm

# --- STYLE CONFIGURATION ---
def matplotlib_default_config():
    font = {'family' : 'serif', 'weight' : 'normal', 'size' : 14}
    rcnew = {
        "mathtext.fontset" : "cm", "text.usetex": False, 'figure.frameon': True,
        'axes.linewidth': 2., "axes.titlesize" : 20, "axes.labelsize" : 20,
        "legend.fontsize" : 12, 'legend.fancybox': True, 'lines.linewidth': 2.5,
        'xtick.direction': 'in', 'ytick.direction': 'in',
        'xtick.top': True, 'ytick.right': True,
        'xtick.labelsize': 18, 'ytick.labelsize': 18
    }
    return font, rcnew

%matplotlib widget 
font, rcnew = matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

In [ ]:
# ==============================================================================
# CELL 1: DATA LOADING & PRE-PROCESSING
# ==============================================================================
# Path points to the newly generated TNG50-4 randomized pickle file
#FILE_FID = '/home/vroberto/FakeSpectra_DLAs/data_TNG/TNG50-4_z3_300Halos_DLAMode_Randomized.pkl'
# Path points to the stratified pickle file
FILE_FID = '/home/vroberto/FakeSpectra_DLAs/notebooks_TNG/data_for_variance_50_4_mask/TNG50-4_z3_637Halos_DLAMode_Stratified_run0_seed42.pkl'
print("Loading DataFrame...")
df_fid = pd.read_pickle(FILE_FID)

# Convert Code Mass to Physical Solar Masses (Arepo/TNG standard 1e10 M_sun / h)
HUBBLE_H = 0.6774
df_fid['halo_mass_msun'] = df_fid['halo_mass'] * (1e10 / HUBBLE_H)

# Clean missing data (spectra that failed extraction or are completely empty)
df_clean = df_fid.dropna(subset=['halo_mass_msun', 'eq_width', 'log_n_hi']).copy()

# Group impact parameters
df_clean['r_mult_round'] = df_clean['r_multiplier'].round(2)
unique_radii = np.sort(df_clean['r_mult_round'].unique())
n_panels = len(unique_radii)

print(f"Total valid skewers: {len(df_clean)}")
print(f"Impact parameters detected: {unique_radii}")

# ==============================================================================
# CELL 2: NATIVE COLUMN DENSITY vs HALO MASS (SCATTER & PDF)
# ==============================================================================
fig_nhi, axes_nhi = plt.subplots(n_panels, 2, figsize=(12, 3 * n_panels), sharey=True, 
                                 gridspec_kw={'width_ratios': [5, 1], 'wspace': 0.0, 'hspace': 0.08})
if n_panels == 1: axes_nhi = np.array([axes_nhi])

# Refined color setup (DLA region slightly darker for better contrast)
c_forest = '#E3F2FD'  # Soft Blue
c_lls    = '#FFF3E0'  # Soft Orange
c_dla    = '#FFD1D8'  # Slightly darker Soft Red/Pink

for i, b_val in enumerate(unique_radii):
    ax_main, ax_hist = axes_nhi[i, 0], axes_nhi[i, 1]
    
    # 1. Backgrounds
    for ax in [ax_main, ax_hist]:
        ax.axhspan(12.0, 17.2, color=c_forest, zorder=0)
        ax.axhspan(17.2, 20.3, color=c_lls, zorder=0)
        ax.axhspan(20.3, 26.0, color=c_dla, zorder=0)
        ax.axhline(17.2, color='white', ls='--', lw=1.5, zorder=0)
        ax.axhline(20.3, color='white', ls='--', lw=1.5, zorder=0)
        ax.grid(True, ls=':', alpha=0.4)

    # 2. Data & Statistics
    df_b = df_clean[df_clean['r_mult_round'] == b_val]
    df_valid = df_b[df_b['log_n_hi'] > 1.0] # Ignore completely empty skewers for mean
    
    ax_main.scatter(df_b['halo_mass_msun'], df_b['log_n_hi'], s=8, alpha=0.2, color='tab:red', zorder=1)
    
    if len(df_valid) > 0:
        bins = np.logspace(np.log10(df_valid['halo_mass_msun'].min()), np.log10(df_valid['halo_mass_msun'].max()), 15)
        means, edges, _ = binned_statistic(df_valid['halo_mass_msun'], df_valid['log_n_hi'], 'mean', bins=bins)
        stds, _, _ = binned_statistic(df_valid['halo_mass_msun'], df_valid['log_n_hi'], 'std', bins=bins)
        centers = np.sqrt(edges[:-1] * edges[1:])
        
        ax_main.errorbar(centers, means, yerr=stds, fmt='ks-', markersize=4, capsize=3, zorder=5, label='Mean $\pm 1\sigma$')
        
        # PDF Plot (Now with filled area)
        if len(df_valid['log_n_hi']) > 1:
            kde = gaussian_kde(df_valid['log_n_hi'])
            y_grid = np.linspace(12.0, 26.0, 100)
            kde_vals = kde(y_grid)
            ax_hist.plot(kde_vals, y_grid, color='darkred', lw=2, zorder=3)
            ax_hist.fill_betweenx(y_grid, 0, kde_vals, color='tab:red', alpha=0.3, zorder=2)

    # 3. Aesthetics & Percentages
    ax_hist.set_xticks([])
    ax_main.set_xscale('log')
    if i < n_panels - 1: ax_main.tick_params(labelbottom=False)
    
    # Calculate HCD percentages
    tot = len(df_b)
    if tot > 0:
        p_dla = len(df_b[df_b['log_n_hi'] >= 20.3]) / tot * 100
        p_lls = len(df_b[(df_b['log_n_hi'] >= 17.2) & (df_b['log_n_hi'] < 20.3)]) / tot * 100
        p_for = len(df_b[(df_b['log_n_hi'] > 1.0) & (df_b['log_n_hi'] < 17.2)]) / tot * 100
        p_emp = len(df_b[df_b['log_n_hi'] <= 1.0]) / tot * 100
    else:
        p_dla = p_lls = p_for = p_emp = 0.0
    
    # Text box for impact parameter and No HCD
    ax_main.text(0.02, 0.85, rf"$b \approx {b_val:.2f} R_{{200}}$" + f"\nNo HCD: {p_for+p_emp:.1f}%", 
                 transform=ax_main.transAxes, bbox=dict(facecolor='white', alpha=0.8, edgecolor='lightgray'), va='top')
    
    # Percentages in the PDF panel
    trans = ax_hist.get_yaxis_transform()
    text_bg = dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1)
    
    if p_dla > 0.1: ax_hist.text(0.9, 23.1, f"{p_dla:.1f}%", transform=trans, ha='right', color='darkred', fontweight='bold', bbox=text_bg)
    if p_lls > 0.1: ax_hist.text(0.9, 18.7, f"{p_lls:.1f}%", transform=trans, ha='right', color='saddlebrown', fontweight='bold', bbox=text_bg)
    if p_for > 0.1: ax_hist.text(0.9, 14.6, f"{p_for:.1f}%", transform=trans, ha='right', color='navy', fontweight='bold', bbox=text_bg)
    
    if i == 0:
        ax_main.set_title(r"Native $\log N_{\rm HI}$ vs. Halo Mass")
        ax_hist.set_title("PDF")
        handles, _ = ax_main.get_legend_handles_labels()
        handles += [patches.Patch(color=c_forest, label='Forest'), 
                    patches.Patch(color=c_lls, label='LLS/Sub-DLA'), 
                    patches.Patch(color=c_dla, label='DLA')]
        ax_main.legend(handles=handles, loc='lower right', ncol=2, frameon=True)

# Global Axis Labels
axes_nhi[-1, 0].set_xlabel(r"Halo Mass ($M_{200}$) [$M_\odot$]")
fig_nhi.text(0.02, 0.5, r"True Simulation $\log N_{\rm HI}$ [cm$^{-2}$]", va='center', rotation='vertical', fontsize=20)
axes_nhi[-1, 0].set_xlim(df_clean['halo_mass_msun'].min()*0.8, df_clean['halo_mass_msun'].max()*1.2)
axes_nhi[-1, 0].set_ylim(12.0, 26.0)

plt.show()

In [ ]:
# ==============================================================================
# CELL 3: ABSORBER COVERING FRACTION vs IMPACT PARAMETER
# ==============================================================================
frac_dla, frac_lls, frac_nohcd = [], [], []

for b_val in unique_radii:
    df_b = df_clean[df_clean['r_mult_round'] == b_val]
    tot = len(df_b)
    if tot > 0:
        frac_dla.append(len(df_b[df_b['log_n_hi'] >= 20.3]) / tot)
        frac_lls.append(len(df_b[(df_b['log_n_hi'] >= 17.2) & (df_b['log_n_hi'] < 20.3)]) / tot)
        frac_nohcd.append(len(df_b[df_b['log_n_hi'] < 17.2]) / tot) # Includes empty skewers
    else:
        frac_dla.append(0.0)
        frac_lls.append(0.0)
        frac_nohcd.append(0.0)

fig_cov, ax_cov = plt.subplots(figsize=(9, 5.5))

# Plot straight lines connecting the data points directly
ax_cov.plot(unique_radii, frac_dla, color='darkred', lw=2.5, marker='o', ms=4, mec='white', 
            label=r'DLA ($\log N_{\mathrm{HI}} \geq 20.3$)')

ax_cov.plot(unique_radii, frac_lls, color='saddlebrown', lw=2.5, marker='s', ms=4, mec='white', 
            label=r'LLS/Sub-DLA ($17.2 \leq \log N_{\mathrm{HI}} < 20.3$)')

ax_cov.plot(unique_radii, frac_nohcd, color='midnightblue', lw=2.5, marker='D', ms=4, mec='white', 
            label=r'No HCD ($\log N_{\mathrm{HI}} < 17.2$)')

ax_cov.grid(True, ls='--', alpha=0.5)
ax_cov.set_xlim(-0.01, unique_radii.max() + 0.1)
ax_cov.set_ylim(-0.02, 1.05)
ax_cov.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

ax_cov.set_xlabel(r'Impact Parameter $b \ [R_{200}]$')
ax_cov.set_ylabel(r'Covering Fraction $f_C(b)$')
ax_cov.legend(loc='center right')

plt.tight_layout()
plt.show()

# ==============================================================================
# CELL 4: PHYSICS CONSTANTS & VOIGT PROFILE
# ==============================================================================
LAMBDA_REST_A = 1215.670100
F_OSC = 0.416400
A21_HZ = 6.265e8
C_KM_S = const.c / 1000.0

def full_voigt_approximation(wave_A, log_N_HI, v_shift_kms, T_gas_K):
    """Calculates F/F0 using the complete Voigt profile via Faddeeva (wofz)."""
    lambda0 = LAMBDA_REST_A * 1e-10 
    N = (10**log_N_HI) * 1e4 
    T_gas_K = max(T_gas_K, 1.0)
    
    b_doppler = np.sqrt(2 * const.k * T_gas_K / const.m_p) 
    v_shifted = (const.c * (wave_A * 1e-10 - lambda0) / lambda0) - (v_shift_kms * 1000.0)
    
    u = v_shifted / b_doppler
    a = (A21_HZ * lambda0) / (4 * np.pi * b_doppler)
    
    phi_v = np.real(wofz(u + 1j * a)) / (np.sqrt(np.pi) * b_doppler) 
    sigma_v = ((const.e**2 * F_OSC) / (4 * const.epsilon_0 * const.m_e * const.c)) * lambda0 * phi_v 
    
    return np.exp(-N * sigma_v)

In [ ]:
# ==============================================================================
# CELL 5: SINGLE HALO VOIGT FIT INSPECTION
# ==============================================================================
# Dynamically select the first available halo ID from the randomized dataset to avoid KeyErrors
SELECTED_HALO_ID = df_clean['halo_id'].unique()[0]
SKEWER_TO_PLOT = 1

df_halo = df_clean[(df_clean['halo_id'] == SELECTED_HALO_ID) & (df_clean['skewer_id'] == SKEWER_TO_PLOT)]

fig_insp, axes_insp = plt.subplots(n_panels * 2, 1, figsize=(8, 3.5 * n_panels), sharex=True, 
                                   gridspec_kw={'height_ratios': [3, 1] * n_panels})
fig_insp.subplots_adjust(hspace=0.08)

for i, b_val in enumerate(unique_radii):
    ax_main, ax_res = axes_insp[i*2], axes_insp[i*2 + 1]
    row = df_halo[df_halo['r_mult_round'] == b_val]
    
    if not row.empty:
        w_data, f_data, sim_nhi = row.iloc[0][['wave', 'flux', 'log_n_hi']]
        v_axis = C_KM_S * (w_data - LAMBDA_REST_A) / LAMBDA_REST_A
        
        # Fit
        guess = 20.0 if np.min(f_data) < 0.05 else 14.0  
        try:
            popt, _ = curve_fit(full_voigt_approximation, w_data, f_data, p0=[guess, 0.0, 2e4], 
                                bounds=([11.0, -1000.0, 1e2], [27.0, 1000.0, 1e12]))
            f_fit = full_voigt_approximation(w_data, *popt)
            fit_nhi, fit_t = popt[0], popt[2]
        except RuntimeError:
            f_fit, fit_nhi, fit_t = np.ones_like(f_data), np.nan, np.nan
        
        # Plot
        ax_main.step(v_axis, f_data, label=rf'Sim ($\log N={sim_nhi:.2f}$)', color='tab:blue', alpha=0.7)
        ax_main.plot(v_axis, f_fit, label=rf'Fit ($\log N={fit_nhi:.2f}$, $T={fit_t:.1e}$K)', color='tab:red', ls='--')
        ax_res.step(v_axis, f_data - f_fit, color='black', alpha=0.6)
            
    ax_main.grid(True, ls=':', alpha=0.5); ax_res.grid(True, ls=':', alpha=0.5)
    ax_main.set_ylim(-0.05, 1.05); ax_res.set_ylim(-0.5, 0.5)
    ax_main.text(0.02, 0.1, rf"$b \approx {b_val} R_{{200}}$", transform=ax_main.transAxes, bbox=dict(facecolor='white'))
    ax_main.legend(loc='lower right', frameon=False, fontsize=10)

axes_insp[0].set_title(f"Voigt DLA Fit | Halo ID: {SELECTED_HALO_ID}")
axes_insp[-1].set_xlabel(r"Velocity $\Delta v$ [km/s]")
axes_insp[-1].set_xlim(-2000, 2000)
plt.show()

In [ ]:
# ==============================================================================
# CELL 6: PARALLEL BATCH FITTING
# ==============================================================================
print(f"Starting parallel batch fitting for {len(df_clean)} spectra...")
t0 = time.time()

bounds = ([11.0, -1000.0, 0], [27.0, 1000.0, 1e12])

def fit_spectrum(row):
    f_data, w_data = row['flux'], row['wave']
    if np.min(f_data) > 0.99: # Noise rejection for empty forest regions
        return {**row.to_dict(), 'fit_log_nhi': np.nan, 'fit_T_gas': np.nan, 'fit_v_shift': np.nan}
    
    guess = 20.0 if np.min(f_data) < 0.05 else 14.0  
    try:
        popt, _ = curve_fit(full_voigt_approximation, w_data, f_data, p0=[guess, 0.0, 2e4], bounds=bounds, maxfev=2000)
        return {**row.to_dict(), 'fit_log_nhi': popt[0], 'fit_v_shift': popt[1], 'fit_T_gas': popt[2]}
    except RuntimeError:
        return {**row.to_dict(), 'fit_log_nhi': np.nan, 'fit_T_gas': np.nan, 'fit_v_shift': np.nan}

# Execute
rows = [row for _, row in df_clean.iterrows()]
with concurrent.futures.ProcessPoolExecutor(max_workers=20) as executor:
    results = list(tqdm(executor.map(fit_spectrum, rows), total=len(rows), desc="Fitting"))

df_fits = pd.DataFrame(results).dropna(subset=['fit_log_nhi', 'fit_T_gas']).copy()
print(f"Fitting complete in {(time.time()-t0)/60:.1f} min. Success: {len(df_fits)}/{len(df_clean)}")

# ==============================================================================
# CELL 7: FITTED COLUMN DENSITY vs HALO MASS
# ==============================================================================
fig_fit, axes_fit = plt.subplots(n_panels, 2, figsize=(12, 3 * n_panels), sharey=True, 
                                 gridspec_kw={'width_ratios': [5, 1], 'wspace': 0.0, 'hspace': 0.08})
if n_panels == 1: axes_fit = np.array([axes_fit])

for i, b_val in enumerate(unique_radii):
    ax_main, ax_hist = axes_fit[i, 0], axes_fit[i, 1]
    
    # 1. Backgrounds
    for ax in [ax_main, ax_hist]:
        ax.axhspan(12.0, 17.2, color=c_forest, zorder=0)
        ax.axhspan(17.2, 20.3, color=c_lls, zorder=0)
        ax.axhspan(20.3, 26.0, color=c_dla, zorder=0)
        ax.axhline(17.2, color='white', ls='--', lw=1.5, zorder=0)
        ax.axhline(20.3, color='white', ls='--', lw=1.5, zorder=0)
        ax.grid(True, ls=':', alpha=0.4)

    # 2. Data & Statistics
    df_b = df_fits[df_fits['r_mult_round'] == b_val]
    masses, nhis = df_b['halo_mass_msun'].values, df_b['fit_log_nhi'].values
    
    ax_main.scatter(masses, nhis, s=10, alpha=0.2, color='tab:blue', marker='s', zorder=1)
    
    if len(masses) > 0:
        bins = np.logspace(np.log10(masses.min()), np.log10(masses.max()), 15)
        means, edges, _ = binned_statistic(masses, nhis, 'mean', bins=bins)
        stds, _, _ = binned_statistic(masses, nhis, 'std', bins=bins)
        centers = np.sqrt(edges[:-1] * edges[1:])
        
        ax_main.errorbar(centers, means, yerr=stds, fmt='ks-', capsize=3, markersize=8, zorder=5, label='Fitted Mean $\pm 1\sigma$')
        
        if len(nhis) > 1:
            # PDF Plot with filled area
            kde = gaussian_kde(nhis)
            y_grid = np.linspace(12.0, 26.0, 100)
            kde_vals = kde(y_grid)
            ax_hist.plot(kde_vals, y_grid, color='midnightblue', lw=2, zorder=3)
            ax_hist.fill_betweenx(y_grid, 0, kde_vals, color='tab:blue', alpha=0.3, zorder=2)

    # 3. Aesthetics & Percentages
    ax_main.set_xscale('log')
    ax_hist.set_xticks([])
    if i < n_panels - 1: ax_main.tick_params(labelbottom=False)
    
    # Calculate HCD percentages (using df_clean to account for skipped/failed fits)
    df_native_b = df_clean[df_clean['r_mult_round'] == b_val]
    tot = len(df_native_b)
    
    if tot > 0:
        p_dla = len(df_b[df_b['fit_log_nhi'] >= 20.3]) / tot * 100
        p_lls = len(df_b[(df_b['fit_log_nhi'] >= 17.2) & (df_b['fit_log_nhi'] < 20.3)]) / tot * 100
        p_for = len(df_b[(df_b['fit_log_nhi'] > 1.0) & (df_b['fit_log_nhi'] < 17.2)]) / tot * 100
        n_failed = tot - len(df_b) # Empty skewers that were skipped by the fitter
        p_nohcd = ((len(df_b[df_b['fit_log_nhi'] < 17.2]) + n_failed) / tot) * 100
    else:
        p_dla = p_lls = p_for = p_nohcd = 0.0
    
    # Text box for impact parameter and No HCD
    ax_main.text(0.02, 0.85, rf"$b \approx {b_val:.2f} R_{{200}}$" + f"\nNo HCD: {p_nohcd:.1f}%", 
                 transform=ax_main.transAxes, bbox=dict(facecolor='white', alpha=0.8, edgecolor='lightgray'), va='top')
    
    # Percentages in the PDF panel
    trans = ax_hist.get_yaxis_transform()
    text_bg = dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1)
    
    if p_dla > 0.1: ax_hist.text(0.9, 23.1, f"{p_dla:.1f}%", transform=trans, ha='right', color='darkred', fontweight='bold', bbox=text_bg)
    if p_lls > 0.1: ax_hist.text(0.9, 18.7, f"{p_lls:.1f}%", transform=trans, ha='right', color='saddlebrown', fontweight='bold', bbox=text_bg)
    if p_for > 0.1: ax_hist.text(0.9, 14.6, f"{p_for:.1f}%", transform=trans, ha='right', color='navy', fontweight='bold', bbox=text_bg)
    
    if i == 0:
        ax_main.set_title("Fitted Voigt $\log N_{\mathrm{HI}}$ vs. Mass")
        ax_hist.set_title("PDF")
        handles, _ = ax_main.get_legend_handles_labels()
        handles += [patches.Patch(color=c_forest, label='Forest'), 
                    patches.Patch(color=c_lls, label='LLS/Sub-DLA'), 
                    patches.Patch(color=c_dla, label='DLA')]
        ax_main.legend(handles=handles, loc='lower right', ncol=2, frameon=True)

axes_fit[-1, 0].set_xlabel(r"Halo Mass ($M_{200}$) [$M_\odot$]")
axes_fit[-1, 0].set_xlim(df_fits['halo_mass_msun'].min()*0.8, df_fits['halo_mass_msun'].max()*1.2)
axes_fit[-1, 0].set_ylim(12.0, 26.0) 

plt.show()

In [ ]:
# ==============================================================================
# CELL 8: COMPARISON - NATIVE vs FITTED COLUMN DENSITY
# ==============================================================================
fig_comp, axes_comp = plt.subplots(n_panels, 2, figsize=(12, 3 * n_panels), sharey=True, 
                                   gridspec_kw={'width_ratios': [5, 1], 'wspace': 0.0, 'hspace': 0.08})
if n_panels == 1: axes_comp = np.array([axes_comp])

for i, b_val in enumerate(unique_radii):
    ax_main, ax_hist = axes_comp[i, 0], axes_comp[i, 1]
    
    for ax in [ax_main, ax_hist]:
        ax.axhspan(12.0, 17.2, color=c_forest, zorder=0)
        ax.axhspan(17.2, 20.3, color=c_lls, zorder=0)
        ax.axhspan(20.3, 26.0, color=c_dla, zorder=0)
        ax.grid(True, ls=':', alpha=0.4)

    df_b = df_fits[df_fits['r_mult_round'] == b_val]
    masses = df_b['halo_mass_msun'].values
    nhis_fit = df_b['fit_log_nhi'].values
    mask_val = df_b['log_n_hi'] > 1.0
    nhis_nat = df_b['log_n_hi'].values
    
    ax_main.scatter(masses[mask_val], nhis_nat[mask_val], s=10, alpha=0.15, color='tab:red', zorder=1)
    ax_main.scatter(masses, nhis_fit, s=10, alpha=0.15, color='tab:blue', marker='s', zorder=2)
    
    if len(masses) > 0:
        bins = np.logspace(np.log10(masses.min()), np.log10(masses.max()), 15)
        centers = np.sqrt(bins[:-1] * bins[1:])
        
        means_nat, _, _ = binned_statistic(masses[mask_val], nhis_nat[mask_val], 'mean', bins=bins)
        ax_main.plot(centers, means_nat, 'darkred', lw=2.5, zorder=5)
        
        means_fit, _, _ = binned_statistic(masses, nhis_fit, 'mean', bins=bins)
        ax_main.plot(centers, means_fit, 'midnightblue', ls='--', lw=2.5, zorder=6)

    y_grid = np.linspace(12.0, 26.0, 100)
    if np.sum(mask_val) > 1:
        kde_nat = gaussian_kde(nhis_nat[mask_val])
        ax_hist.plot(kde_nat(y_grid), y_grid, 'darkred', lw=2)
    if len(nhis_fit) > 1:
        kde_fit = gaussian_kde(nhis_fit)
        ax_hist.plot(kde_fit(y_grid), y_grid, 'midnightblue', ls='--', lw=2)
    
    ax_main.set_xscale('log')
    ax_hist.set_xticks([])
    if i < n_panels - 1: ax_main.tick_params(labelbottom=False)
    
    ax_main.text(0.02, 0.85, rf"$b \approx {b_val:.2f} R_{{200}}$", transform=ax_main.transAxes, bbox=dict(facecolor='white', alpha=0.8))
    
    if i == 0:
        ax_main.set_title("Comparison: Native vs Fitted")
        custom = [Line2D([0],[0], color='darkred', lw=2, label='Mean Native'), Line2D([0],[0], color='midnightblue', lw=2, ls='--', label='Mean Fit')]
        ax_main.legend(handles=custom, loc='lower right')

axes_comp[-1, 0].set_xlabel(r"Halo Mass ($M_{200}$) [$M_\odot$]")
axes_comp[-1, 0].set_xlim(df_fits['halo_mass_msun'].min()*0.8, df_fits['halo_mass_msun'].max()*1.2)
axes_comp[-1, 0].set_ylim(12.0, 26.0) 
plt.show()

In [ ]:
# ==============================================================================
# CELL 9: FITTED TEMPERATURE vs HALO MASS
# ==============================================================================
fig_temp, axes_temp = plt.subplots(n_panels, 2, figsize=(12, 3 * n_panels), sharey=True, 
                                   gridspec_kw={'width_ratios': [5, 1], 'wspace': 0.0, 'hspace': 0.08})
if n_panels == 1: axes_temp = np.array([axes_temp])

for i, b_val in enumerate(unique_radii):
    ax_main, ax_hist = axes_temp[i, 0], axes_temp[i, 1]
    
    df_b = df_fits[df_fits['r_mult_round'] == b_val]
    # Filter unphysical lower bounds (T ~ 100K artifact)
    mask = df_b['fit_T_gas'] > 150.0
    masses, temps = df_b['halo_mass_msun'].values[mask], df_b['fit_T_gas'].values[mask]
    
    ax_main.scatter(masses, temps, s=10, alpha=0.3, color='tab:orange', zorder=1)
    
    if len(masses) > 0:
        bins = np.logspace(np.log10(masses.min()), np.log10(masses.max()), 15)
        log_t = np.log10(temps)
        
        means, edges, _ = binned_statistic(masses, log_t, 'mean', bins=bins)
        stds, _, _ = binned_statistic(masses, log_t, 'std', bins=bins)
        centers = np.sqrt(edges[:-1] * edges[1:])
        
        yerr_low = 10**means - 10**(means - stds)
        yerr_up = 10**(means + stds) - 10**means
        ax_main.errorbar(centers, 10**means, yerr=[yerr_low, yerr_up], fmt='k.-', zorder=5)
        
        # KDE in log space
        if len(temps) > 1:
            kde = gaussian_kde(log_t)
            y_grid_log = np.linspace(3.0, 11.0, 100)
            ax_hist.plot(kde(y_grid_log), 10**y_grid_log, color='saddlebrown', lw=2)

    for ax in [ax_main, ax_hist]:
        ax.grid(True, ls=':', alpha=0.5)
        ax.set_yscale('log')
        
    ax_main.set_xscale('log')
    ax_hist.set_xticks([])
    if i < n_panels - 1: ax_main.tick_params(labelbottom=False)
    
    ax_main.text(0.02, 0.85, rf"$b \approx {b_val:.2f} R_{{200}}$", transform=ax_main.transAxes, bbox=dict(facecolor='white', alpha=0.8))
    if i == 0: ax_main.set_title("Fitted Gas Temperature vs. Mass")

axes_temp[-1, 0].set_xlabel(r"Halo Mass ($M_{200}$) [$M_\odot$]")
axes_temp[-1, 0].set_xlim(df_fits['halo_mass_msun'].min()*0.8, df_fits['halo_mass_msun'].max()*1.2)
axes_temp[-1, 0].set_ylim(1e3, 1e11)
plt.show()